# 01 · Data Collection

Loads the curated fund universe (36+ Indian mutual fund schemes with AMFI codes), fetches NAV history from AMFI/mfapi (with deterministic synthetic fallback), and fetches benchmark indices from Yahoo Finance.

**Data sources**
- AMFI India / mfapi.in — daily NAV history per scheme code
- Yahoo Finance — `^NSEI`, `^NSMIDCP`, `^GSPC`, gold/silver ETFs
- Curated CSV — fund metadata (TER, AUM, management fee, loads)

In [ ]:
# Bootstrap path so the notebook finds the src/ package
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)


In [ ]:
from data_loader import load_all, LoaderConfig, load_fund_universe

universe = load_fund_universe()
print(f"Total schemes: {len(universe)}")
universe.head()

### Category mix

In [ ]:
universe["category"].value_counts()

### Fetch NAV history & benchmarks

Set `offline_mode=False` to pull live data; `True` uses the deterministic synthetic generator so the notebook always completes.

In [ ]:
data = load_all(LoaderConfig(offline_mode=True, lookback_days=1260, use_cache=True))
navs, benchmarks = data["navs"], data["benchmarks"]
print(f"NAV rows: {len(navs):,}  ·  unique schemes: {navs.scheme_code.nunique()}")
print(f"Benchmark rows: {len(benchmarks):,}  ·  series: {benchmarks.benchmark.nunique()}")

### Plot a few NAV series

In [ ]:
import matplotlib.pyplot as plt

sample = navs[navs.scheme_code.isin(universe.scheme_code.sample(5, random_state=1))]
fig, ax = plt.subplots(figsize=(10, 4))
for code, grp in sample.groupby("scheme_code"):
    name = universe.loc[universe.scheme_code == code, "scheme_name"].values[0]
    ax.plot(grp.date, grp.nav / grp.nav.iloc[0], label=name[:38], lw=1.2)
ax.set_title("Sample normalized NAV paths")
ax.legend(fontsize=7, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()